In [ ]:
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(".."))
import config
import matplotlib.pyplot as plt

In [ ]:
config.path_to_evaluation

In [ ]:
metrics = ["NRMSE", "MASE", "sMAPE"]
p_list = ["5000", "12000", "25000", "50000", "100000", "all"]
feats = ['abs_energy','intermittency','mean','median','kurtosis','skewness','standard_deviation','agg_autocorrelation_max','erraticness','agg_linear_trend_slope']

In [ ]:
mmd = pd.read_parquet(f"{config.path_to_evaluation}/mmd_distances")
fid = pd.read_parquet(f"{config.path_to_evaluation}/fid_similarity.parquet")
ks = pd.read_parquet(f"{config.path_to_evaluation}/1-KS_similarity.parquet")

In [ ]:
metrics = pd.merge(mmd, fid, on = ["run", "div", "p"])
metrics = pd.merge(metrics, ks, on= ["run", "div", "p"])
metrics

Plot median mmd distance, fid distance and 1-KS similarity over runs and div steps together

In [ ]:
median_df = (
    metrics.groupby("p")[["mmd", "fid", "ks"]]
    .median()
    .reset_index()
)

p_order = ["5000", "12000", "25000", "50000", "100000", "all"]
median_df["p"] = pd.Categorical(
    median_df["p"],
    categories=p_order,
    ordered=True,
)
median_df = median_df.sort_values("p")

fig, ax1 = plt.subplots(figsize=(8, 5))

ax2 = ax1.twinx()
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("outward", 50))

# Colors
c_mmd = "tab:blue"
c_fid = "tab:orange"
c_ks = "tab:green"

ax1.plot(
    median_df["p"],
    median_df["mmd"],
    marker="o",
    linewidth=2,
    color=c_mmd,
)

ax2.plot(
    median_df["p"],
    median_df["fid"],
    marker="s",
    linewidth=2,
    color=c_fid,
)

ax3.plot(
    median_df["p"],
    median_df["ks"],
    marker="^",
    linewidth=2,
    color=c_ks,
)

# clean axes
for ax in [ax1, ax2, ax3]:
    ax.set_ylabel("")
    ax.set_yticks([])

ax1.spines["left"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax3.spines["right"].set_visible(False)

# direct labels
ax1.annotate(
    "MMD",
    xy=(len(median_df) - 1, median_df["mmd"].iloc[-1]),
    xytext=(10, 0),
    textcoords="offset points",
    color=c_mmd,
    fontsize=11,
    fontweight="bold",
    va="center",
)

ax2.annotate(
    "FID",
    xy=(len(median_df) - 1, median_df["fid"].iloc[-1]),
    xytext=(10, 0),
    textcoords="offset points",
    color=c_fid,
    fontsize=11,
    fontweight="bold",
    va="center",
)

ax3.annotate(
    "KS",
    xy=(len(median_df) - 1, median_df["ks"].iloc[-1]),
    xytext=(10, 0),
    textcoords="offset points",
    color=c_ks,
    fontsize=11,
    fontweight="bold",
    va="center",
)

ax1.set_xlabel("Potential target size", fontsize=14)
ax1.margins(x=0.1)
ax1.set_ylabel("Source-target distance (FID,MMD)\n/similarity (KS)", fontsize=14)

plt.tight_layout()
plt.savefig(f"{config.path_to_evaluation}/similarity_measure_comparison.pdf")
plt.show()